In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [2]:
df = pd.read_csv("restaurant_sales.csv")


In [3]:
df['Date'] = pd.to_datetime(df['Date'])
df['DayOfWeek'] = df['Date'].dt.dayofweek


In [4]:
df

,Date,Day,Weather,Holiday,Customers,Sales,DayOfWeek
0,2024-01-01,Monday,Snowy,Yes,84,8736,0
1,2024-01-02,Tuesday,Rainy,Yes,168,18984,1
2,2024-01-03,Wednesday,Sunny,No,114,15504,2
3,2024-01-04,Thursday,Cloudy,Yes,124,16616,3
4,2024-01-05,Friday,Cloudy,Yes,151,16006,4
...,...,...,...,...,...,...,...
360,2024-12-26,Thursday,Snowy,No,183,25254,3
361,2024-12-27,Friday,Sunny,Yes,227,25424,4
362,2024-12-28,Saturday,Snowy,Yes,196,29400,5
363,2024-12-29,Sunday,Cloudy,No,97,13580,6


In [5]:
df['Weather'] = df['Weather'].astype('category').cat.codes


In [7]:
df

,Date,Day,Weather,Holiday,Customers,Sales,DayOfWeek
0,2024-01-01,Monday,2,Yes,84,8736,0
1,2024-01-02,Tuesday,1,Yes,168,18984,1
2,2024-01-03,Wednesday,3,No,114,15504,2
3,2024-01-04,Thursday,0,Yes,124,16616,3
4,2024-01-05,Friday,0,Yes,151,16006,4
...,...,...,...,...,...,...,...
360,2024-12-26,Thursday,2,No,183,25254,3
361,2024-12-27,Friday,3,Yes,227,25424,4
362,2024-12-28,Saturday,2,Yes,196,29400,5
363,2024-12-29,Sunday,0,No,97,13580,6


In [8]:
df['Holiday'] = df['Holiday'].map({'Yes': 1, 'No': 0})


In [9]:
df

,Date,Day,Weather,Holiday,Customers,Sales,DayOfWeek
0,2024-01-01,Monday,2,1,84,8736,0
1,2024-01-02,Tuesday,1,1,168,18984,1
2,2024-01-03,Wednesday,3,0,114,15504,2
3,2024-01-04,Thursday,0,1,124,16616,3
4,2024-01-05,Friday,0,1,151,16006,4
...,...,...,...,...,...,...,...
360,2024-12-26,Thursday,2,0,183,25254,3
361,2024-12-27,Friday,3,1,227,25424,4
362,2024-12-28,Saturday,2,1,196,29400,5
363,2024-12-29,Sunday,0,0,97,13580,6


In [11]:
X = df[['DayOfWeek', 'Weather', 'Holiday', 'Customers']]
y = df['Sales']


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [15]:
from sklearn.ensemble import RandomForestRegressor


In [17]:
model = RandomForestRegressor(n_estimators=100, random_state=42)


In [18]:
model.fit(X_train, y_train)


RandomForestRegressor(random_state=42)

In [19]:
preds = model.predict(X_test)


In [34]:
preds

array([31077.41      ,  6818.86190476,  5005.63      , 26807.16      ,
       25861.73      , 21371.345     , 13789.01      , 22021.84      ,
       11052.03      , 26246.672     ,  9931.49      , 19443.818     ,
        9830.4       , 28243.85      , 18318.2085    , 15931.75      ,
       10405.47      , 12432.52      , 25685.56      , 16845.77666667,
       24910.64      ,  9109.2       , 10495.57      , 26951.93      ,
       16238.86666667, 15716.46      , 18487.36      , 14216.2       ,
       18325.905     , 28792.35      , 16840.56      , 20603.77      ,
       16577.44      ,  8378.7       , 21844.14      , 18928.42      ,
       22130.915     , 12798.44      , 15320.56      , 11743.35      ,
       21823.61      , 24935.2       , 16244.64666667, 21759.0425    ,
       14279.94      ,  8929.49      , 20554.03      , 12592.24      ,
       16353.44      , 15070.22666667, 11239.51      ,  9476.87      ,
       18823.11      , 21448.305     , 21417.19      , 25241.38      ,
      

In [21]:
from sklearn.metrics import mean_squared_error
import math

In [23]:
rmse = math.sqrt(mean_squared_error(y_test, preds))


In [24]:
print(f"Model RMSE: {rmse:.2f}")


Model RMSE: 2261.96


In [26]:
future_dates = pd.date_range(start=df['Date'].max() + pd.Timedelta(days=1), periods=30)
future_data = pd.DataFrame({
    'Date': future_dates,
    'DayOfWeek': future_dates.dayofweek,
    'Weather': np.random.randint(0, 4, size=30),  # Random weather codes
    'Holiday': np.random.randint(0, 2, size=30),  # Random holiday flags
    'Customers': np.random.randint(80, 200, size=30)  # Estimated customers
})

In [27]:
future_sales = model.predict(future_data[['DayOfWeek', 'Weather', 'Holiday', 'Customers']])
future_data['PredictedSales'] = future_sales



In [29]:
future_sales

array([10920.11      , 21210.8       , 21570.01      , 21573.29785714,
       28107.85      , 12100.43      , 19075.93      , 15214.98      ,
       15567.15      , 24209.28535714, 18725.25      , 13938.29      ,
        9619.57      , 19103.755     , 20248.255     , 14459.48666667,
       23487.98      , 12508.45      , 12989.94      , 20535.82      ,
       24489.46      , 18927.31      , 12594.98      , 16306.47166667,
       10173.57      , 23469.56285714, 10897.87      , 19746.28      ,
       15381.77      , 21953.33      ])

In [30]:
avg_customers_per_staff = 30
future_data['StaffNeeded'] = future_data['Customers'].apply(lambda x: math.ceil(x / avg_customers_per_staff))

In [32]:
future_data.to_csv("staff_plan.csv", index=False)
print("Future predictions and staff recommendations saved to 'staff_plan.csv'.")

Future predictions and staff recommendations saved to 'staff_plan.csv'.
